# Obarvování černobílých krajin

Tento notebook natrénuje neuronovou síť, která doplní barvy do černobílé fotky krajiny.

Jak to funguje: vezmeme barevné fotky krajin a uděláme z nich černobílé. Síť se pak učí,
jakou barvu kam doplnit. Nemusíme nic ručně označovat, stačí běžné barevné fotky.

Použijeme model typu GAN. Jsou v něm dvě sítě. První (generátor) barvu doplňuje. Druhá
(diskriminátor) hádá, jestli výsledek vypadá jako opravdová fotka. Generátor se snaží
druhou síť přesvědčit, a proto volí živější a věrohodnější barvy.

## 1. Příprava

Načteme knihovny a nastavíme náhodný seed, aby šel běh zopakovat se stejným
výsledkem. Kód si sám zvolí, na čem bude počítat. Pokud je k dispozici grafická karta,
použije ji pro rychlost.

Přepínač FAST_MODE určuje velikost běhu. Když je zapnutý, hodí se hlavně na vyzkoušení i na procesoru.
Když se vypne, model je větší, ale potřebuje grafickou kartu.

In [ ]:
from __future__ import annotations

import platform
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from skimage import color as skcolor
from torch import nn
from torch.utils.data import DataLoader, Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True   # robustnost vůči poškozeným JPEG

SEED = 42
FAST_MODE = True


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Python:      {platform.python_version()}")
print(f"PyTorch:     {torch.__version__}")
print(f"Zařízení:    {DEVICE}")

In [ ]:
@dataclass(frozen=True)
class Config:
    # FAST_MODE = malé hodnoty pro rychlou kontrolu na CPU. Vypnout pro plný běh na GPU.
    image_size: int = 32 if FAST_MODE else 96
    train_images: int = 800 if FAST_MODE else 3_500       # Landscape Pictures má ~4 300 obrázků
    val_images: int = 200 if FAST_MODE else 600
    batch_size: int = 32
    generator_channels: int = 16 if FAST_MODE else 64      # šířka generátoru (U-Net)
    discriminator_channels: int = 16 if FAST_MODE else 64  # šířka diskriminátoru
    learning_rate: float = 2e-4
    adam_beta1: float = 0.5
    colour_match_weight: float = 100.0                     # jak silně se držet skutečné barvy
    epochs: int = 20 if FAST_MODE else 60


config = Config()
config

## 2. Data: složka s fotkami krajin

Stáhněte si sadu fotek krajin **Landscape Pictures** (asi 4 300 fotek) a rozbalte ji do složky `./data/landscapes`. Dataset najdete na Kaggle:
<https://www.kaggle.com/datasets/arnaud58/landscape-pictures>.

Každou fotku převedeme do barevného prostoru Lab. Z něj vezmeme jas (kanál `L`) jako vstup
a barvu (kanály `a`, `b`) jako to, co se má síť naučit. Tenhle převod je pomalý, proto ho
uděláme jen jednou a výsledky si necháme v paměti.

In [ ]:
def lab_to_rgb(lightness, colours):
    """Z normalizovaných tenzorů jasu (1,H,W) a barvy (2,H,W) složí RGB obrázek (H,W,3)."""
    lightness_values = (lightness.detach().cpu().numpy()[0] + 1.0) * 50.0
    colour_values = colours.detach().cpu().numpy() * 110.0
    lab_image = np.stack([lightness_values, colour_values[0], colour_values[1]], axis=2).astype(np.float64)
    return np.clip(skcolor.lab2rgb(lab_image), 0.0, 1.0)


def image_to_lab_tensors(image, image_size):
    """Jeden PIL obrázek -> (jas (1,H,W), barva (2,H,W)) v normalizovaném Lab."""
    image = image.convert("RGB").resize((image_size, image_size), Image.BILINEAR)
    rgb_pixels = np.asarray(image, dtype=np.float32) / 255.0
    lab_image = skcolor.rgb2lab(rgb_pixels).astype(np.float32)
    lightness = lab_image[:, :, 0:1] / 50.0 - 1.0
    colours = lab_image[:, :, 1:3] / 110.0
    return (torch.from_numpy(lightness).permute(2, 0, 1),
            torch.from_numpy(colours).permute(2, 0, 1))


class ColorizationFolderDataset(Dataset):
    """Načte zadané obrázky a převede je na (jas, barva) v Lab.
    rgb2lab spočítá jednou a uloží do paměti. Poškozené soubory přeskočí."""

    def __init__(self, paths, image_size):
        lightness_list, colours_list = [], []
        skipped_count = 0
        for path in paths:
            try:
                image = Image.open(path)
                lightness, colours = image_to_lab_tensors(image, image_size)
            except Exception:
                skipped_count += 1
                continue
            lightness_list.append(lightness)
            colours_list.append(colours)
        if not lightness_list:
            raise RuntimeError("Nepodařilo se načíst žádný obrázek.")
        if skipped_count:
            print(f"  (přeskočeno {skipped_count} nečitelných souborů)")
        self.lightness = torch.stack(lightness_list)
        self.colours = torch.stack(colours_list)

    def __len__(self):
        return self.lightness.size(0)

    def __getitem__(self, index):
        return self.lightness[index], self.colours[index]

In [ ]:
DATA_DIR = "./data/landscapes"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_paths = sorted(p for p in Path(DATA_DIR).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
if not all_paths:
    raise FileNotFoundError(
        f"V {DATA_DIR} nejsou žádné obrázky. Stáhněte dataset (viz buňka výše) "
        f"a rozbalte ho do {DATA_DIR}/."
    )

random_generator = torch.Generator().manual_seed(SEED)
shuffled_indices = torch.randperm(len(all_paths), generator=random_generator).tolist()
total_needed = config.train_images + config.val_images
train_paths = [all_paths[index] for index in shuffled_indices[: config.train_images]]
val_paths = [all_paths[index] for index in shuffled_indices[config.train_images : total_needed]]

print(f"Nalezeno {len(all_paths)} obrázků | použito train/val: "
      f"{len(train_paths)}/{len(val_paths)}")
print("Předpočítávám Lab tenzory (jednorázově)…")
train_dataset = ColorizationFolderDataset(train_paths, config.image_size)
val_dataset = ColorizationFolderDataset(val_paths, config.image_size)
print("Hotovo.")

train_loader = DataLoader(
    train_dataset, batch_size=config.batch_size, shuffle=True,
    generator=torch.Generator().manual_seed(SEED), drop_last=True,
)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

print(f"Trénovacích: {len(train_dataset)} | validačních: {len(val_dataset)}")

## 3. Generátor

Generátor z jasu (1 kanál) vyrobí barvu (2 kanály). Má tvar zvaný U-Net. Nejdřív obrázek
postupně zmenšuje a zjišťuje, co na něm je. Potom ho zase zvětšuje zpátky na původní
velikost. Mezi oběma částmi vedou propojky, které pomáhají udržet ostré hrany.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, features):
        return self.layers(features)


class UNet(nn.Module):
    """Generátor: z jasu (1 kanál) udělá barvu (2 kanály). Velikost obrázku musí být dělitelná 8."""

    def __init__(self, base_channels=64):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.encoder1 = DoubleConv(1, base_channels)
        self.encoder2 = DoubleConv(base_channels, base_channels * 2)
        self.encoder3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)
        self.upsample3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.decoder3 = DoubleConv(base_channels * 8, base_channels * 4)
        self.upsample2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.decoder2 = DoubleConv(base_channels * 4, base_channels * 2)
        self.upsample1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.decoder1 = DoubleConv(base_channels * 2, base_channels)
        self.output_layer = nn.Conv2d(base_channels, 2, 1)

    def forward(self, lightness):
        skip1 = self.encoder1(lightness)
        skip2 = self.encoder2(self.pool(skip1))
        skip3 = self.encoder3(self.pool(skip2))
        bottleneck = self.bottleneck(self.pool(skip3))
        decoded = self.decoder3(torch.cat([self.upsample3(bottleneck), skip3], dim=1))
        decoded = self.decoder2(torch.cat([self.upsample2(decoded), skip2], dim=1))
        decoded = self.decoder1(torch.cat([self.upsample1(decoded), skip1], dim=1))
        return torch.tanh(self.output_layer(decoded))


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

## 4. Diskriminátor

Diskriminátor je druhá síť. Dostane jas spolu s barvou a hádá, jestli to vypadá jako
opravdová fotka. Nehodnotí obrázek najednou, ale po malých kouscích. Díky tomu musí barvy
sedět v každé části obrázku, ne jen celkově.

In [ ]:
class PatchDiscriminator(nn.Module):
    """Druhá síť. Dostane jas + barvu (3 kanály) a po malých kouscích
    říká, jak moc každý kousek vypadá jako z opravdové fotky."""

    def __init__(self, in_channels=3, base_filters=64):
        super().__init__()
        # obrázek postupně zmenšujeme a přidáváme kanály
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, base_filters, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_filters, base_filters * 2, 4, stride=2, padding=1),
            nn.BatchNorm2d(base_filters * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_filters * 2, base_filters * 4, 4, stride=2, padding=1),
            nn.BatchNorm2d(base_filters * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_filters * 4, base_filters * 8, 4, stride=1, padding=1),
            nn.BatchNorm2d(base_filters * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # poslední vrstva dá jedno číslo pro každý kousek obrázku
            nn.Conv2d(base_filters * 8, 1, 4, stride=1, padding=1),
        )

    def forward(self, lightness, colours):
        return self.layers(torch.cat([lightness, colours], dim=1))


set_seed()
generator = UNet(base_channels=config.generator_channels).to(DEVICE)
discriminator = PatchDiscriminator(in_channels=3, base_filters=config.discriminator_channels).to(DEVICE)

print(f"Generátor (U-Net):        {count_parameters(generator):,} parametrů")
print(f"Diskriminátor (PatchGAN): {count_parameters(discriminator):,} parametrů")

# kontrola tvarů
example_lightness, example_colours = next(iter(train_loader))
example_lightness = example_lightness.to(DEVICE)
example_colours = example_colours.to(DEVICE)
with torch.no_grad():
    example_output = generator(example_lightness)
    example_score = discriminator(example_lightness, example_output)
print(f"Generátor:     {tuple(example_lightness.shape)} -> {tuple(example_output.shape)}")
print(f"Diskriminátor: (jas, barva) -> mřížka {tuple(example_score.shape)}")

## 5. Trénink

Obě sítě se učí zároveň a soupeří spolu. V každém kroku se stane tohle:

1. Diskriminátor se zlepšuje v tom, aby poznal opravdovou barvu od té vymyšlené.
2. Generátor se učí dvě věci najednou. Vyrobit barvu, která diskriminátor zmate, a zároveň
   zůstat blízko skutečné barvy.

Ta druhá část drží barvy při zemi a celý trénink uklidňuje.

In [ ]:
@torch.no_grad()
def compute_validation_loss(generator, loader):
    generator.eval()
    criterion = nn.L1Loss()
    total_loss, seen_count = 0.0, 0
    for lightness, colours in loader:
        lightness, colours = lightness.to(DEVICE), colours.to(DEVICE)
        total_loss += criterion(generator(lightness), colours).item() * lightness.size(0)
        seen_count += lightness.size(0)
    return total_loss / seen_count


def train_gan(generator, discriminator, train_loader, val_loader, config):
    adversarial_criterion = nn.MSELoss()      # jak dobře diskriminátor pozná pravé od falešných
    l1_criterion = nn.L1Loss()                # jak blízko jsou barvy ke skutečným
    generator_optimizer = torch.optim.Adam(
        generator.parameters(), lr=config.learning_rate, betas=(config.adam_beta1, 0.999))
    discriminator_optimizer = torch.optim.Adam(
        discriminator.parameters(), lr=config.learning_rate, betas=(config.adam_beta1, 0.999))
    history = []
    report_every = max(1, config.epochs // 10)

    for epoch in range(1, config.epochs + 1):
        generator.train(); discriminator.train()
        running_discriminator_loss = 0.0
        running_adversarial_loss = 0.0
        running_l1_loss = 0.0
        seen_count = 0

        for lightness, colours in train_loader:
            lightness, colours = lightness.to(DEVICE), colours.to(DEVICE)

            # ---- Diskriminátor ----
            fake_colours = generator(lightness).detach()
            real_score = discriminator(lightness, colours)
            fake_score = discriminator(lightness, fake_colours)
            discriminator_loss = 0.5 * (
                adversarial_criterion(real_score, torch.ones_like(real_score)) +
                adversarial_criterion(fake_score, torch.zeros_like(fake_score)))
            discriminator_optimizer.zero_grad()
            discriminator_loss.backward()
            discriminator_optimizer.step()

            # ---- Generátor ----
            fake_colours = generator(lightness)
            fake_score = discriminator(lightness, fake_colours)
            generator_adversarial_loss = adversarial_criterion(fake_score, torch.ones_like(fake_score))
            generator_colour_loss = l1_criterion(fake_colours, colours) * config.colour_match_weight
            generator_loss = generator_adversarial_loss + generator_colour_loss
            generator_optimizer.zero_grad()
            generator_loss.backward()
            generator_optimizer.step()

            batch = lightness.size(0)
            running_discriminator_loss += discriminator_loss.item() * batch
            running_adversarial_loss += generator_adversarial_loss.item() * batch
            running_l1_loss += generator_colour_loss.item() * batch
            seen_count += batch

        validation_loss = compute_validation_loss(generator, val_loader)
        history.append({
            "epoch": epoch,
            "discriminator_loss": running_discriminator_loss / seen_count,
            "generator_adversarial_loss": running_adversarial_loss / seen_count,
            "generator_colour_loss": running_l1_loss / seen_count,
            "validation_loss": validation_loss,
        })
        if epoch == 1 or epoch % report_every == 0 or epoch == config.epochs:
            print(f"epoch {epoch:>3}/{config.epochs} | "
                  f"diskriminátor {running_discriminator_loss/seen_count:.3f} | "
                  f"generátor {running_adversarial_loss/seen_count:.3f} | "
                  f"barva {running_l1_loss/seen_count:.3f} | validace {validation_loss:.4f}")

    return pd.DataFrame(history)

In [ ]:
set_seed()
generator = UNet(base_channels=config.generator_channels).to(DEVICE)
discriminator = PatchDiscriminator(in_channels=3, base_filters=config.discriminator_channels).to(DEVICE)
history = train_gan(generator, discriminator, train_loader, val_loader, config)

## 6. Výsledky

Podíváme se na průběh učení a na pár obarvených obrázků vedle sebe.

Druhý graf ukazuje, jak daleko jsou odhadnuté barvy od skutečných. Čím níž, tím líp.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(history["epoch"], history["discriminator_loss"], label="diskriminátor")
ax[0].plot(history["epoch"], history["generator_adversarial_loss"], label="generátor")
ax[0].set_title("Ztráty při soupeření sítí"); ax[0].set_xlabel("epocha")
ax[0].legend(); ax[0].grid(alpha=0.2)
ax[1].plot(history["epoch"], history["validation_loss"], color="tab:green")
ax[1].set_title("Jak daleko jsou barvy od pravdy"); ax[1].set_xlabel("epocha")
ax[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def show_predictions(generator, dataset, num_examples=6):
    generator.eval()
    fig, axes = plt.subplots(num_examples, 3, figsize=(8, 2.6 * num_examples))
    titles = ["vstup (černobílá)", "GAN predikce", "pravda"]
    for row in range(num_examples):
        lightness, colours = dataset[row]
        predicted_colours = generator(lightness.unsqueeze(0).to(DEVICE))[0].cpu()
        panels = [lab_to_rgb(lightness, torch.zeros_like(colours)),
                  lab_to_rgb(lightness, predicted_colours),
                  lab_to_rgb(lightness, colours)]
        for column in range(3):
            axes[row, column].imshow(panels[column]); axes[row, column].axis("off")
            if row == 0:
                axes[row, column].set_title(titles[column])
    plt.tight_layout()
    plt.show()


show_predictions(generator, val_dataset, num_examples=6)

## 7. Obarvení vlastní fotky

Teď můžeme obarvit jakoukoli černobílou fotku. Síť odhadne barvu na zmenšené verzi. Tu pak
zvětšíme a spojíme s jasem v plné velikosti. Detaily proto zůstanou ostré a obrázek si
udrží svoji velikost.

In [ ]:
@torch.no_grad()
def colorize_image(path, generator, image_size=None, save=True):
    image_size = image_size or config.image_size
    generator.eval()

    grayscale = Image.open(path).convert("L")
    width, height = grayscale.size

    gray_full = np.asarray(grayscale, dtype=np.float32) / 255.0
    lightness_full = skcolor.rgb2lab(np.stack([gray_full] * 3, axis=2))[:, :, 0]

    small = grayscale.resize((image_size, image_size), Image.BILINEAR)
    gray_small = np.asarray(small, dtype=np.float32) / 255.0
    lightness_small = skcolor.rgb2lab(np.stack([gray_small] * 3, axis=2))[:, :, 0].astype(np.float32)
    model_input = torch.from_numpy(lightness_small / 50.0 - 1.0)[None, None].to(DEVICE)

    predicted_colours_small = (generator(model_input)[0].cpu().numpy() * 110.0).astype(np.float32)
    predicted_colours_full = np.stack(
        [np.asarray(Image.fromarray(predicted_colours_small[channel], mode="F").resize(
            (width, height), Image.BILINEAR), dtype=np.float32) for channel in range(2)],
        axis=2,
    )

    lab_image = np.concatenate([lightness_full[:, :, None], predicted_colours_full], axis=2).astype(np.float64)
    rgb_image = np.clip(skcolor.lab2rgb(lab_image), 0.0, 1.0)

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(gray_full, cmap="gray"); ax[0].set_title("vstup (černobílá)"); ax[0].axis("off")
    ax[1].imshow(rgb_image); ax[1].set_title("obarveno (GAN)"); ax[1].axis("off")
    plt.tight_layout()
    plt.show()

    if save:
        output_path = Path(path).with_name(Path(path).stem + "_gan_colorized.png")
        Image.fromarray((rgb_image * 255).astype(np.uint8)).save(output_path)
        print(f"Uloženo: {output_path}")
    return rgb_image

In [ ]:
demo_image = Image.open(val_paths[0]).convert("RGB")
demo_path = "demo_gray.png"
demo_image.convert("L").save(demo_path)

colorize_image(demo_path, generator)

plt.figure(figsize=(4, 4))
plt.imshow(demo_image); plt.title("skutečná barva"); plt.axis("off")
plt.show()

### Výběr fotky z počítače

Po spuštění další buňky se otevře okno, ve kterém vyberete fotku. Síť ji rovnou obarví.

In [ ]:
def pick_and_colorize(generator):
    """Otevře okno pro výběr obrázku a obarví ho."""
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception as error:
        print("tkinter není dostupné:", error)
        return None
    window = tk.Tk()
    window.withdraw()
    window.attributes("-topmost", True)
    path = filedialog.askopenfilename(
        title="Vyberte černobílou fotografii",
        filetypes=[("Obrázky", "*.png *.jpg *.jpeg *.bmp *.tif *.tiff"),
                   ("Všechny soubory", "*.*")],
    )
    window.destroy()
    if not path:
        print("Nebyl vybrán žádný soubor.")
        return None
    print("Vybráno:", path)
    return colorize_image(path, generator)


pick_and_colorize(generator)

## 8. Poznámky

- Barvy jsou živější než u jednoduché verze. Soupeření obou sítí totiž tlačí na věrohodný
  vzhled.
- Když se barvy začnou kazit, je čas zkusit menší rychlost učení nebo
  dát větší váhu na shodu se skutečnou barvou.
- Na pěkný výsledek pusťte plný běh na grafické kartě (vypněte `FAST_MODE`). Na procesoru
  slouží `FAST_MODE` jen k ověření, že kód běží.
- Model umí obarvit hlavně to, co viděl při tréninku. Když ho učíte na krajinách, krajiny
  mu půjdou dobře, jiné motivy hůř.